In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# --- Gold bootstrap: paths, imports, audit timestamp ---

import notebookutils
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, IntegerType, LongType, DoubleType,
    StringType, TimestampType, DateType, BooleanType
)
from datetime import datetime, timezone, date

# Lakehouse paths
SILVER_LH = notebookutils.lakehouse.getWithProperties("silver")["properties"]["abfsPath"]
GOLD_LH = notebookutils.lakehouse.getWithProperties("gold")["properties"]["abfsPath"]

# Source (Silver)
SILVER_TRIPS = f"{SILVER_LH}/Tables/yellow_tripdata"
SILVER_ZONES = f"{SILVER_LH}/Tables/taxi_zone_lookup"

# Destinations (Gold)
GOLD_FACT_TRIP = f"{GOLD_LH}/Tables/FactTrip"
GOLD_DIM_DATE = f"{GOLD_LH}/Tables/DimDate"
GOLD_DIM_VENDOR = f"{GOLD_LH}/Tables/DimVendor"
GOLD_DIM_LOCATION = f"{GOLD_LH}/Tables/DimLocation"
GOLD_DIM_PAYMENT = f"{GOLD_LH}/Tables/DimPaymentType"

processing_ts = datetime.now(timezone.utc)

print(f"Silver source: {SILVER_TRIPS}")
print(f"Gold tables base: {GOLD_LH}/Tables/")
print(f"Processing timestamp: {processing_ts.isoformat()}")

StatementMeta(, 3ab1b14f-a3ca-4cda-9cc6-973bc9fccfa1, 3, Finished, Available, Finished, False)

Silver source: abfss://73a7dd19-3e75-431d-9f13-0dbc0f72f9c3@onelake.dfs.fabric.microsoft.com/4186ae61-ab44-4157-bcc5-a94295cd9cf6/Tables/yellow_tripdata
Gold tables base: abfss://73a7dd19-3e75-431d-9f13-0dbc0f72f9c3@onelake.dfs.fabric.microsoft.com/c8793b39-b8f5-4a76-b44c-7876ca633c8e/Tables/
Processing timestamp: 2026-05-27T20:02:50.167147+00:00


In [3]:
# --- DimDate ---
date_start = date(2023, 1, 1)
date_end = date(2025, 12, 31)

dim_date = (
    spark.sql(f"""
        SELECT explode(sequence(
            to_date('{date_start}'),
            to_date('{date_end}'),
            interval 1 day
        )) AS full_date
    """)
    .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast(IntegerType()))
    .withColumn("year", F.year("full_date"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("month_short", F.date_format("full_date", "MMM"))
    .withColumn("day", F.dayofmonth("full_date"))
    .withColumn("day_of_week", F.dayofweek("full_date"))
    .withColumn("day_name", F.date_format("full_date", "EEEE"))
    .withColumn("week_of_year", F.weekofyear("full_date"))
    .withColumn("is_weekend", F.col("day_of_week").isin([1, 7]))
    .withColumn("year_month", F.date_format("full_date", "yyyy-MM"))
    .withColumn("_processed_at", F.lit(processing_ts).cast(TimestampType()))
)

dim_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_DIM_DATE)
print(f"✓ DimDate: {dim_date.count()} rows")

StatementMeta(, 3ab1b14f-a3ca-4cda-9cc6-973bc9fccfa1, 5, Finished, Available, Finished, False)

✓ DimDate: 1096 rows


In [4]:
# --- DimVendor ---
dim_vendor = spark.createDataFrame([
    (1, 1, "Creative Mobile Technologies", "CMT"),
    (2, 2, "VeriFone Inc.", "VFI"),
    (3, 6, "Myle Technologies", "MYLE"),
    (4, 7, "Helix", "HELIX"),
], ["vendor_key", "vendor_id", "vendor_name", "vendor_short_code"])

dim_vendor = dim_vendor.withColumn("_processed_at", F.lit(processing_ts).cast(TimestampType()))

dim_vendor.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_DIM_VENDOR)
print(f"✓ DimVendor: {dim_vendor.count()} rows")

StatementMeta(, 3ab1b14f-a3ca-4cda-9cc6-973bc9fccfa1, 6, Finished, Available, Finished, False)

✓ DimVendor: 4 rows


In [5]:
# --- DimLocation ---
silver_zones = spark.read.format("delta").load(SILVER_ZONES)

window_loc = Window.orderBy("LocationID")

dim_location = (
    silver_zones
    .withColumn("location_key", F.row_number().over(window_loc))
    .withColumnRenamed("LocationID", "location_id")
    .withColumnRenamed("Borough", "borough")
    .withColumnRenamed("Zone", "zone_name")
    .select("location_key", "location_id", "borough", "zone_name", "service_zone")
    .withColumn("_processed_at", F.lit(processing_ts).cast(TimestampType()))
)

# Sentinel Unknown row
unknown_loc = spark.createDataFrame(
    [(-1, -1, "Unknown", "Unknown", "Unknown")],
    ["location_key", "location_id", "borough", "zone_name", "service_zone"]
).withColumn("_processed_at", F.lit(processing_ts).cast(TimestampType()))

dim_location = dim_location.unionByName(unknown_loc)

dim_location.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_DIM_LOCATION)
print(f"✓ DimLocation: {dim_location.count()} rows")

StatementMeta(, 3ab1b14f-a3ca-4cda-9cc6-973bc9fccfa1, 7, Finished, Available, Finished, False)

✓ DimLocation: 266 rows


In [6]:
# --- DimPaymentType ---
dim_payment = spark.createDataFrame([
    (0, 0, "Flex Fare"),
    (1, 1, "Credit card"),
    (2, 2, "Cash"),
    (3, 3, "No charge"),
    (4, 4, "Dispute"),
    (5, 5, "Unknown"),
    (6, 6, "Voided trip"),
], ["payment_key", "payment_type_id", "payment_type_name"])

dim_payment = dim_payment.withColumn("_processed_at", F.lit(processing_ts).cast(TimestampType()))

dim_payment.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_DIM_PAYMENT)
print(f"✓ DimPaymentType: {dim_payment.count()} rows")

StatementMeta(, 3ab1b14f-a3ca-4cda-9cc6-973bc9fccfa1, 8, Finished, Available, Finished, False)

✓ DimPaymentType: 7 rows


In [7]:
# --- DimPaymentType ---
dim_payment = spark.createDataFrame([
    (0, 0, "Flex Fare"),
    (1, 1, "Credit card"),
    (2, 2, "Cash"),
    (3, 3, "No charge"),
    (4, 4, "Dispute"),
    (5, 5, "Unknown"),
    (6, 6, "Voided trip"),
], ["payment_key", "payment_type_id", "payment_type_name"])

dim_payment = dim_payment.withColumn("_processed_at", F.lit(processing_ts).cast(TimestampType()))

dim_payment.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_DIM_PAYMENT)
print(f"✓ DimPaymentType: {dim_payment.count()} rows")

StatementMeta(, 3ab1b14f-a3ca-4cda-9cc6-973bc9fccfa1, 9, Finished, Available, Finished, False)

✓ DimPaymentType: 7 rows


In [8]:
# --- FactTrip: join silver to all 4 dimensions, materialize surrogate keys ---

# Re-read silver and dimensions (fresh, in case anything was cached stale)
silver_df = spark.read.format("delta").load(SILVER_TRIPS)
dim_date = spark.read.format("delta").load(GOLD_DIM_DATE).select("date_key", "full_date")
dim_vendor = spark.read.format("delta").load(GOLD_DIM_VENDOR).select("vendor_key", "vendor_id")
dim_location = spark.read.format("delta").load(GOLD_DIM_LOCATION).select("location_key", "location_id")
dim_payment = spark.read.format("delta").load(GOLD_DIM_PAYMENT).select("payment_key", "payment_type_id")

# Aliases so we can join the same dim twice (pickup loc + dropoff loc, pickup date + dropoff date)
dim_loc_pu = dim_location.alias("pu_loc")
dim_loc_do = dim_location.alias("do_loc")
dim_date_pu = dim_date.alias("pu_date")
dim_date_do = dim_date.alias("do_date")

fact_trip = (
    silver_df
    # Cast pickup/dropoff to dates for date dimension joins
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumn("dropoff_date", F.to_date("tpep_dropoff_datetime"))
    # Join DimDate (pickup)
    .join(dim_date_pu, F.col("pickup_date") == F.col("pu_date.full_date"), "left")
    .withColumnRenamed("date_key", "pickup_date_key")
    .drop("full_date")
    # Join DimDate (dropoff)
    .join(dim_date_do, F.col("dropoff_date") == F.col("do_date.full_date"), "left")
    .withColumnRenamed("date_key", "dropoff_date_key")
    .drop("full_date")
    # Join DimVendor
    .join(dim_vendor, silver_df["VendorID"] == dim_vendor["vendor_id"], "left")
    .drop("vendor_id")
    # Join DimLocation (pickup)
    .join(dim_loc_pu, F.col("PULocationID") == F.col("pu_loc.location_id"), "left")
    .withColumnRenamed("location_key", "pickup_location_key")
    .drop("location_id")
    # Join DimLocation (dropoff)
    .join(dim_loc_do, F.col("DOLocationID") == F.col("do_loc.location_id"), "left")
    .withColumnRenamed("location_key", "dropoff_location_key")
    .drop("location_id")
    # Join DimPaymentType
    .join(dim_payment, silver_df["payment_type"] == dim_payment["payment_type_id"], "left")
    .drop("payment_type_id")
    # Coalesce any orphan location FKs to the -1 sentinel
    .withColumn("pickup_location_key", F.coalesce(F.col("pickup_location_key"), F.lit(-1)))
    .withColumn("dropoff_location_key", F.coalesce(F.col("dropoff_location_key"), F.lit(-1)))
    # Final column selection — keep surrogate keys + measures, drop natural keys
    .select(
        # Surrogate keys (FKs to dimensions)
        "pickup_date_key",
        "dropoff_date_key",
        "vendor_key",
        "pickup_location_key",
        "dropoff_location_key",
        "payment_key",
        # Measures
        "passenger_count",
        "trip_distance",
        "trip_duration_seconds",
        "fare_amount",
        "extra",
        "mta_tax",
        "tip_amount",
        "tolls_amount",
        "improvement_surcharge",
        "total_amount",
        "congestion_surcharge",
        "Airport_fee",
        # Pickup/dropoff timestamps preserved for time-of-day analysis
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        # Flags
        "store_and_fwd_flag",
        "RatecodeID",
        # Partition keys (already on silver)
        "_pickup_year",
        "_pickup_month",
    )
    .withColumn("_gold_processed_at", F.lit(processing_ts).cast(TimestampType()))
)

# Write partitioned FactTrip
(
    fact_trip.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("_pickup_year", "_pickup_month")
    .save(GOLD_FACT_TRIP)
)

fact_count = fact_trip.count()
print(f"✓ FactTrip: {fact_count:,} rows")
print(f"  Expected: 20,073,684 (matches silver)")
print(f"  Match: {fact_count == 20073684}")

StatementMeta(, 3ab1b14f-a3ca-4cda-9cc6-973bc9fccfa1, 10, Finished, Available, Finished, False)

✓ FactTrip: 20,073,684 rows
  Expected: 20,073,684 (matches silver)
  Match: True


In [10]:
# --- Gold validation: reads + row counts ---

fact = spark.read.format("delta").load(GOLD_FACT_TRIP)
dim_date = spark.read.format("delta").load(GOLD_DIM_DATE)
dim_vendor = spark.read.format("delta").load(GOLD_DIM_VENDOR)
dim_location = spark.read.format("delta").load(GOLD_DIM_LOCATION)
dim_payment = spark.read.format("delta").load(GOLD_DIM_PAYMENT)

print("=== Check 1: row counts ===")
print(f"  FactTrip:        {fact.count():>12,}")
print(f"  DimDate:         {dim_date.count():>12,}")
print(f"  DimVendor:       {dim_vendor.count():>12,}")
print(f"  DimLocation:     {dim_location.count():>12,}")
print(f"  DimPaymentType:  {dim_payment.count():>12,}")

StatementMeta(, 3ab1b14f-a3ca-4cda-9cc6-973bc9fccfa1, 12, Finished, Available, Finished, False)

=== Check 1: row counts ===
  FactTrip:          20,073,684
  DimDate:                1,096
  DimVendor:                  4
  DimLocation:              266
  DimPaymentType:             7


In [11]:
print("=== Check 2: no orphan FKs in FactTrip ===")
orphans = fact.select(
    F.count(F.when(F.col("pickup_date_key").isNull(), 1)).alias("null_pickup_date"),
    F.count(F.when(F.col("dropoff_date_key").isNull(), 1)).alias("null_dropoff_date"),
    F.count(F.when(F.col("vendor_key").isNull(), 1)).alias("null_vendor"),
    F.count(F.when(F.col("pickup_location_key").isNull(), 1)).alias("null_pickup_loc"),
    F.count(F.when(F.col("dropoff_location_key").isNull(), 1)).alias("null_dropoff_loc"),
    F.count(F.when(F.col("payment_key").isNull(), 1)).alias("null_payment"),
)
orphans.show(truncate=False)

StatementMeta(, 3ab1b14f-a3ca-4cda-9cc6-973bc9fccfa1, 13, Finished, Available, Finished, False)

=== Check 2: no orphan FKs in FactTrip ===
+----------------+-----------------+-----------+---------------+----------------+------------+
|null_pickup_date|null_dropoff_date|null_vendor|null_pickup_loc|null_dropoff_loc|null_payment|
+----------------+-----------------+-----------+---------------+----------------+------------+
|0               |0                |0          |0              |0               |0           |
+----------------+-----------------+-----------+---------------+----------------+------------+



In [12]:
print("=== Check 3: top 5 pickup locations ===")
fact_with_loc = fact.join(
    dim_location,
    fact["pickup_location_key"] == dim_location["location_key"]
)
fact_with_loc.groupBy("borough", "zone_name").count().orderBy(F.desc("count")).show(5, truncate=False)

StatementMeta(, 3ab1b14f-a3ca-4cda-9cc6-973bc9fccfa1, 14, Finished, Available, Finished, False)

=== Check 3: top 5 pickup locations ===
+---------+---------------------+------+
|borough  |zone_name            |count |
+---------+---------------------+------+
|Manhattan|Midtown Center       |943078|
|Manhattan|Upper East Side South|937276|
|Queens   |JFK Airport          |885729|
|Manhattan|Upper East Side North|872712|
|Manhattan|Midtown East         |693804|
+---------+---------------------+------+
only showing top 5 rows



In [13]:
print("=== Check 4: revenue by vendor ===")
fact_with_vendor = fact.join(
    dim_vendor,
    fact["vendor_key"] == dim_vendor["vendor_key"]
)
fact_with_vendor.groupBy("vendor_name").agg(
    F.count("*").alias("trip_count"),
    F.round(F.sum("total_amount"), 2).alias("total_revenue"),
).orderBy(F.desc("trip_count")).show(truncate=False)

StatementMeta(, 3ab1b14f-a3ca-4cda-9cc6-973bc9fccfa1, 15, Finished, Available, Finished, False)

=== Check 4: revenue by vendor ===
+----------------------------+----------+--------------+
|vendor_name                 |trip_count|total_revenue |
+----------------------------+----------+--------------+
|VeriFone Inc.               |15150739  |4.3339523259E8|
|Creative Mobile Technologies|4921939   |1.3122674611E8|
|Myle Technologies           |1006      |49563.95      |
+----------------------------+----------+--------------+



In [14]:
print("=== Check 5: FactTrip schema ===")
fact.printSchema()

StatementMeta(, 3ab1b14f-a3ca-4cda-9cc6-973bc9fccfa1, 16, Finished, Available, Finished, False)

=== Check 5: FactTrip schema ===
root
 |-- pickup_date_key: integer (nullable = true)
 |-- dropoff_date_key: integer (nullable = true)
 |-- vendor_key: long (nullable = true)
 |-- pickup_location_key: long (nullable = true)
 |-- dropoff_location_key: long (nullable = true)
 |-- payment_key: long (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- trip_duration_seconds: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- store_and_fwd_flag: 